In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')

### Load Cleaned Dataset

In [2]:
movies = pd.read_csv("data/cleaned_movies.csv")

In [3]:
movies.head(2)

,genres,keywords,overview,title,movie_id,cast,crew
0,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...",Avatar,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [4]:
movies.shape

(4806, 7)

### Feature Engineering

#### 1.Extract Genres & Keywords

In [5]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
       L.append(i['name'])
    return L

In [6]:
movies['genres']=movies['genres'].apply(convert)

In [7]:
movies['keywords'] = movies['keywords'].apply(convert)

In [8]:
movies.head(1)

,genres,keywords,overview,title,movie_id,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...",Avatar,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


- Genres and keywords are extracted from JSON-like strings and converted into list format for further processing.

#### 2.Extract Top 3 Cast Members

In [9]:
def convert3(obj):
    L = []
    counter = 0 
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter+=1
        else:
            break


    return L

In [10]:
movies['cast'] = movies['cast'].apply(convert)

In [11]:
movies['cast'].head(2)

0    [Sam Worthington, Zoe Saldana, Sigourney Weave...
1    [Johnny Depp, Orlando Bloom, Keira Knightley, ...
Name: cast, dtype: object

In [12]:
movies.head(1)

,genres,keywords,overview,title,movie_id,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...",Avatar,19995,"[Sam Worthington, Zoe Saldana, Sigourney Weave...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


- Only the top three cast members are retained to reduce noise and improve recommendation relevance.

#### 3.Extract Director Information

In [13]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [14]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [15]:
movies.head(2)

,genres,keywords,overview,title,movie_id,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...",Avatar,19995,"[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron]
1,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...",Pirates of the Caribbean: At World's End,285,"[Johnny Depp, Orlando Bloom, Keira Knightley, ...",[Gore Verbinski]


- Director information is extracted because directors significantly influence movie style and content.

### Text Processing

In [16]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [17]:
movies.head(1)

,genres,keywords,overview,title,movie_id,cast,crew
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,19995,"[Sam Worthington, Zoe Saldana, Sigourney Weave...",[James Cameron]


In [18]:
movies["genres"] = movies["genres"].apply(lambda x:[i.replace(" ","")for i in x])
movies["keywords"] = movies["keywords"].apply(lambda x:[i.replace(" ","")for i in x])
movies["cast"] = movies["cast"].apply(lambda x:[i.replace(" ","")for i in x])
movies["crew"] = movies["crew"].apply(lambda x:[i.replace(" ","")for i in x])

- Text preprocessing ensures consistency by removing spaces and standardizing textual features.

### Create Tags

In [19]:
movies['tags'] =(
    movies['overview'] +
    movies['genres'] + 
    movies['keywords'] + movies['cast'] + movies['crew'])

In [20]:
movies.head(1)

,genres,keywords,overview,title,movie_id,cast,crew,tags
0,"[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[In, the, 22nd, century,, a, paraplegic, Marin...",Avatar,19995,"[SamWorthington, ZoeSaldana, SigourneyWeaver, ...",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [21]:
movies.drop(columns = ["keywords","genres","overview","cast","crew"])

,title,movie_id,tags
0,Avatar,19995,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,285,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,206647,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,49026,"[Following, the, death, of, District, Attorney..."
4,John Carter,49529,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4801,El Mariachi,9367,"[El, Mariachi, just, wants, to, play, his, gui..."
4802,Newlyweds,72766,"[A, newlywed, couple's, honeymoon, is, upended..."
4803,"Signed, Sealed, Delivered",231617,"[""Signed,, Sealed,, Delivered"", introduces, a,..."
4804,Shanghai Calling,126186,"[When, ambitious, New, York, attorney, Sam, is..."


In [22]:
new_df = movies[["title","movie_id","tags"]]

- A single "tags" column is created by combining all important content features. This consolidated representation will be used to measure movie similarity.

In [23]:
new_df

,title,movie_id,tags
0,Avatar,19995,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,285,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,206647,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,49026,"[Following, the, death, of, District, Attorney..."
4,John Carter,49529,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4801,El Mariachi,9367,"[El, Mariachi, just, wants, to, play, his, gui..."
4802,Newlyweds,72766,"[A, newlywed, couple's, honeymoon, is, upended..."
4803,"Signed, Sealed, Delivered",231617,"[""Signed,, Sealed,, Delivered"", introduces, a,..."
4804,Shanghai Calling,126186,"[When, ambitious, New, York, attorney, Sam, is..."


### Text Normalization

In [24]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [25]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [26]:
new_df['tags']

0       in the 22nd century, a paraplegic marine is di...
1       captain barbossa, long believed to be dead, ha...
2       a cryptic message from bond’s past sends him o...
3       following the death of district attorney harve...
4       john carter is a war-weary, former military ca...
                              ...                        
4801    el mariachi just wants to play his guitar and ...
4802    a newlywed couple's honeymoon is upended by th...
4803    "signed, sealed, delivered" introduces a dedic...
4804    when ambitious new york attorney sam is sent t...
4805    ever since the second grade when he first saw ...
Name: tags, Length: 4806, dtype: object

## Stemming

In [27]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [28]:
def stem(text):
    y =[]
    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [29]:
new_df["tags"] = new_df["tags"].apply(stem)

In [30]:
new_df["tags"][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav stephenlang michellerodriguez giovanniribisi joeldavidmoor cchpounder wesstudi lazalonso dileeprao mattgerald seananthonymoran jasonwhyt scottlawr kellykilgour jamespatrickpitt seanpatrickmurphi peterdillon kevindorman kelsonhenderson davidvanhorn jacobtomuri michaelblain-rozgay joncurri lukehawk woodyschultz petermensah soniaye jahnelcurfman ilramchoi kylawarren lisaroumain debrawilson chrismala taylorkibbi jodielandau julielamm cullenb.madden josephbradymadden frankietorr austinwilson sarawilson tamicawashington-mil lucybri nathanmeist gerryblair matthewchamberlain paulyat wraywil

- Stemming converts words to their root forms, reducing redundancy and improving similarity calculations.

### Text Vectorization

In [31]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_df = 5000,stop_words='english')

In [32]:
vectors = cv.fit_transform(new_df["tags"]).toarray()

- CountVectorizer converts textual movie descriptions into numerical feature vectors that can be processed mathematically.

In [33]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 81234))

In [34]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(81234,))

In [35]:
cv.get_feature_names_out()[:100]

array(['00', '000', '007', '07am', '10', '100', '1000', '101', '108',
       '10th', '11', '114', '117', '118', '119', '11th', '12', '1200',
       '1215', '1250', '125th', '12th', '13', '1300', '13th', '14', '140',
       '1408', '142', '1429', '148', '14pm', '14th', '15', '150', '150th',
       '1520', '1536', '15th', '15thcenturi', '16', '1600s', '161',
       '1630s', '1644', '1681', '1691', '16th', '16thcenturi', '17',
       '170', '1700s', '173rd', '1748', '1776', '17th', '17thcenturi',
       '18', '180', '1800', '1818', '1820', '1820s', '1824', '1831',
       '1834', '1836', '1838', '1839', '1841', '1845', '1850', '1856',
       '1857', '1860', '1862', '1863', '1870', '1875', '1876', '1879',
       '1880s', '1882', '1885', '1889', '1890', '18th', '18thcenturi',
       '19', '1900', '1900s', '1903', '1905', '191', '1910', '1911',
       '1912', '1914', '1915', '1917'], dtype=object)

In [36]:
ps.stem('loved')

'love'

### Similarity Matrix

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

In [38]:
similarity = cosine_similarity(vectors)

- Cosine Similarity measures the similarity between movie vectors. Movies with higher similarity scores are considered more relevant recommendations.

### Recommendation Function

In [39]:
def recommend(movie):
    movie_index = new_df[new_df["title"] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

    
    

### Testing Recommendation Engine

In [40]:
recommend("Avatar")

Titan A.E.
Predator
Aliens
Aliens vs Predator: Requiem
Predators


- A content-based movie recommendation system was successfully developed using feature engineering, NLP preprocessing, CountVectorizer, and Cosine Similarity.